# Tutorial 3: Train NicheTrans on SMA data

In [1]:
import os, time, datetime, warnings

import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from model.nicheTrans_img import *
from datasets.data_manager_SMA import SMA

from utils.utils import *
from utils.utils_training_SMA import train, test
from utils.utils_dataloader import *

warnings.filterwarnings("ignore")

/home/ubuntu/miniconda3/envs/NicheTrans/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Initialize the args and fix seeds

In [2]:
%run ./args/args_SMA.py
args = args

set_seed(args.seed)
if torch.cuda.is_available():
    os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device: {}".format(device))
print("==========\nArgs:{}\n==========".format(args))

Using device: cpu
Args:Namespace(noise_rate=0.2, dropout_rate=0.1, n_source=3000, n_target=50, img_size=256, workers=4, path_img='/mnt/datadisk0/Processed_DATA/2023_nbt_SMA/Processed_data_used/patches', rna_path='/mnt/datadisk0/Processed_DATA/2023_nbt_SMA/Processed_data_used', msi_path='/mnt/datadisk0/Processed_DATA/2023_nbt_SMA/Processed_data_used', max_epoch=40, stepsize=20, train_batch=32, test_batch=32, optimizer='adam', lr=0.0003, gamma=0.1, weight_decay=0.0005, seed=1, save_dir='./log', eval_step=1, gpu_devices='0', context_model_type='transformer', gnn_num_layers=2, gnn_hidden_dim=None, gnn_graph_type='full')


### Initialize dataloaders and NicheTrans

In [3]:
# create the dataloaders
dataset = SMA(path_img=args.path_img, rna_path=args.rna_path, msi_path=args.msi_path, n_top_genes=args.n_source, n_top_targets=args.n_target)
trainloader, testloader = sma_dataloader(args, dataset)

# create the model
source_dimension, target_dimension = dataset.rna_length, dataset.msi_length
model = NicheTrans(source_length=source_dimension, target_length=target_dimension, noise_rate=args.noise_rate, dropout_rate=args.dropout_rate,gnn_graph_type='star')
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)

------Calculating spatial graph...


The graph contains 12134 edges, 3120 cells.
3.8891 neighbors per cell on average.


------Calculating spatial graph...


The graph contains 24190 edges, 3120 cells.
7.7532 neighbors per cell on average.


------Calculating spatial graph...


...

The graph contains 22578 edges, 2918 cells.
7.7375 neighbors per cell on average.


------Calculating spatial graph...
The graph contains 10360 edges, 2675 cells.
3.8729 neighbors per cell on average.


------Calculating spatial graph...


The graph contains 20628 edges, 2675 cells.
7.7114 neighbors per cell on average.


=> SMA loaded
Dataset statistics:
  ------------------------------
  subset   | # num | 
  ------------------------------
  train    |  Without filtering  6038 spots from     2 slides 
  test     |  Without filtering  2675 spots from     1 slides
  train    |  After filting  6005 spots from     2 slides 
  test     |  After filting  2655 spots from     1 slides
  ------------------------------


### Initialize loss function (criterion) and optimizer

In [4]:
criterion = nn.MSELoss()

if args.optimizer == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
elif args.optimizer == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
else:
    print('unexpected optimizer')

if args.stepsize > 0:
    scheduler = lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)

### Model training and testing

In [5]:
start_time = time.time()

for epoch in range(args.max_epoch):
    last_epoch = epoch + 1 == args.max_epoch

    print("==> Epoch {}/{}".format(epoch+1, args.max_epoch))
    
    ################
    train(model, criterion, optimizer, trainloader, use_img=False, device=device)
    if args.stepsize > 0: scheduler.step()
    ################

pearson = test(model, testloader, use_img=False, device=device)
os.makedirs('./trained_model', exist_ok=True)
torch.save(model.state_dict(), './trained_model/NicheTrans_SMA_last.pth')

elapsed = round(time.time() - start_time)
elapsed = str(datetime.timedelta(seconds=elapsed))
print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))

==> Epoch 1/40


Batch 187/187	 Loss 61.470528 (92.789200)
==> Epoch 2/40


Batch 187/187	 Loss 22.653725 (39.532784)
==> Epoch 3/40


Batch 187/187	 Loss 9.221058 (14.540421)
==> Epoch 4/40


Batch 187/187	 Loss 6.975023 (7.194235)
==> Epoch 5/40


...

Batch 187/187	 Loss 2.920301 (3.381768)
==> Epoch 38/40


Batch 187/187	 Loss 4.510206 (3.345550)
==> Epoch 39/40


Batch 187/187	 Loss 3.340818 (3.315517)
==> Epoch 40/40


Batch 187/187	 Loss 2.721687 (3.318080)


Testing Set: pearson correlation 0.3270 + 0.0735; spearman correlation 0.4111 + 0.0767; rmse 2.4554 + 1.7839
Finished. Total elapsed time (h:m:s): 0:14:56
